# ISLR Phase 2 — Isolated Sign Language Recognition (20 Glosses)

Pipeline nhận diện ngôn ngữ ký hiệu độc lập người ký (Signer-Independent) sử dụng bộ dữ liệu Kaggle `asl-signs`.

**Đặc điểm:**
- **Thin Notebook**: Không chứa business logic, chỉ điều phối qua các CLI scripts chuẩn hóa.
- **Config-driven**: Toàn bộ tham số cấu hình trong `configs/islr.yaml` và cơ chế `--set key=value`.
- **Zero data leakage**: Split theo `participant_id`, kiểm tra nghiêm ngặt không rò rỉ người ký.
- **Đầy đủ kiểm chứng**: Sanity Overfit check (acc >= 95%), Random Forest baseline, và GRU multi-seed.

## 1. Setup Môi trường & Kaggle Credentials

In [ ]:
import os
import sys
from pathlib import Path

# 1. Nhận diện môi trường
in_colab = 'google.colab' in sys.modules
in_kaggle = 'kaggle_web_client' in sys.modules or os.path.exists('/kaggle/working')

if in_colab:
    print("==> Môi trường: Google Colab")
    # Clone hoặc pull repo
    repo_name = "SignLanguageReconition_ML"
    if not os.path.exists(repo_name):
        !git clone -b test/kaggle-training https://github.com/ChienPM-27/SignLanguageReconition_ML.git
    %cd {repo_name}
    !git pull origin test/kaggle-training
    !pip install -q -r requirements-colab.txt

    # Đọc Kaggle Credentials từ Colab User Secrets
    try:
        from google.colab import userdata
        try:
            os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
            print("✓ Đã nạp KAGGLE_API_TOKEN từ Colab Secrets.")
        except Exception:
            os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
            os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
            print("✓ Đã nạp KAGGLE_USERNAME và KAGGLE_KEY từ Colab Secrets.")
    except Exception as e:
        print(f"! Lưu ý Kaggle Secrets: {e}. Nếu chưa cấu hình, vui lòng thêm trong Secrets tab.")

elif in_kaggle:
    print("==> Môi trường: Kaggle Notebook")
    # Kaggle tự động có credentials trong Secrets
else:
    print("==> Môi trường: Local")

# Đảm bảo repo root nằm trong sys.path
repo_root = Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"Working Directory: {repo_root}")

## 2. Cấu hình Nhanh (Overrides)
Chỉ định danh sách `--set` để ghi đè các tham số mong muốn mà không sửa file code.

In [ ]:
# Tùy chọn: Chạy Smoke Test nhanh (3 files/gloss) hoặc Full Dataset (None)
SMOKE_TEST = True

OVERRIDES = []
if SMOKE_TEST:
    OVERRIDES.extend([
        "data.max_files_per_gloss=3",
        "training.epochs=15",
        "training.patience=5",
        "training.seeds=[42]",
    ])

# Biến chuỗi CLI options
OVERRIDE_FLAGS = " ".join(f"--set {opt}" for opt in OVERRIDES)
print(f"CLI Overrides: {OVERRIDE_FLAGS}")

## 3. Step 0: Kiểm tra Kết nối & Cấu hình (Setup Check)

In [ ]:
!python scripts/00_setup_check.py --config configs/islr.yaml {OVERRIDE_FLAGS}

## 4. Step 1-4: Chạy Toàn bộ Pipeline Dữ liệu (Selection -> Download -> Preprocess -> Split)

In [ ]:
!python scripts/10_prepare_data.py --config configs/islr.yaml --stage all {OVERRIDE_FLAGS}

## 5. Step 5a: Sanity Check — Overfit Verification
Huấn luyện 1 mẫu/gloss với dropout=0 và tắt augmentation; kỳ vọng accuracy đạt >= 95%.

In [ ]:
!python scripts/20_train.py --config configs/islr.yaml --overfit-check {OVERRIDE_FLAGS}

## 6. Step 5b: Baseline Thống kê — Random Forest
Huấn luyện Random Forest trên đặc trưng thống kê (mean, std, min, max qua trục thời gian) để làm mốc so sánh.

In [ ]:
!python scripts/20_train.py --config configs/islr.yaml --baseline-rf {OVERRIDE_FLAGS}

## 7. Step 5c: Huấn luyện Mô hình GRU (Config-driven)

In [ ]:
!python scripts/20_train.py --config configs/islr.yaml {OVERRIDE_FLAGS}

## 8. Hiển thị Kết quả & Biểu đồ

In [ ]:
from IPython.display import Image, display
from pathlib import Path

runs_dir = Path("outputs/runs")
if runs_dir.is_dir():
    recent_runs = sorted(runs_dir.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
    if recent_runs:
        latest_run = recent_runs[0]
        print(f"Hiển thị artifacts từ: {latest_run.name}")
        
        # Tìm file ảnh learning curve và confusion matrix
        for img in latest_run.rglob("*.png"):
            print(f"Ảnh: {img.name}")
            display(Image(filename=str(img)))
            
        # In nội dung test_report.json
        for rep in latest_run.rglob("*report*.json"):
            print(f"Báo cáo {rep.name}:")
            print(rep.read_text(encoding='utf-8')[:1000])
else:
    print("Chưa có runs trong outputs/runs")

## 9. Đóng gói & Tải về Kết quả (Artifacts Zip)

In [ ]:
import shutil

archive_name = "islr_phase2_artifacts"
!zip -q -r {archive_name}.zip outputs/ reports/ data/selection/ 2>/dev/null || tar -czf {archive_name}.tar.gz outputs/ reports/ data/selection/

print(f"Đã tạo file nén kết quả. Bạn có thể tải file từ file browser bên trái:")
for ext in [".zip", ".tar.gz"]:
    fpath = Path(f"{archive_name}{ext}")
    if fpath.is_file():
        print(f"- {fpath.name} ({fpath.stat().st_size / 1024:.1f} KB)")